# 1. Data and Tokenization

This notebook loads the dataset, inspects sample traces, and compares the coarse burst tokenizer with the richer proposal-style burst tokenizer.

In [1]:
!python -m pip install -q numpy

In [2]:
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/final project/dataset')
else:
    PROJECT_DIR = Path.cwd()

print(f'Project directory: {PROJECT_DIR}')

Mounted at /content/drive
Project directory: /content/drive/MyDrive/final project/dataset


In [3]:
import warnings
from collections.abc import Callable, Iterable, Iterator, Sequence

import numpy as np

In [4]:
DATASET_NAME = 'tor_100w_2500tr.npz'
N_SITES = 100
N_TRACES = 100

DATASET_PATH = PROJECT_DIR / DATASET_NAME
print(f'Dataset path: {DATASET_PATH}')

Dataset path: /content/drive/MyDrive/final project/dataset/tor_100w_2500tr.npz


In [5]:
def resolve_dataset_path(data_path: str | Path) -> Path:
    path = Path(data_path)

    if path.is_dir():
        subdirs = sorted(
            entry
            for entry in path.iterdir()
            if entry.is_dir() and not entry.name.startswith('__')
        )
        npz_files = sorted(
            entry
            for entry in path.iterdir()
            if entry.is_file() and entry.suffix.lower() == '.npz'
        )

        if subdirs:
            return path
        if npz_files:
            return npz_files[0]
        raise ValueError(f'No dataset directories or .npz files found in {path}')

    if path.is_file() and path.suffix.lower() == '.npz':
        return path

    raise ValueError(f'Dataset path does not exist or is unsupported: {path}')


def load_traces(
    data_path: str | Path,
    n_sites: int = 100,
    n_traces: int = 100,
) -> tuple[list[list[int]], list[int]]:
    resolved_path = resolve_dataset_path(data_path)

    if resolved_path.is_file():
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                category=np.exceptions.VisibleDeprecationWarning,
            )
            with np.load(resolved_path, allow_pickle=True) as dataset:
                data = dataset['data']
                raw_labels = np.asarray(dataset['labels'])

        unique_sites = np.unique(raw_labels)
        selected_sites = unique_sites[:n_sites]
        traces: list[list[int]] = []
        labels: list[int] = []
        site_to_id = {site: index for index, site in enumerate(selected_sites)}

        for site in selected_sites:
            site_indices = np.where(raw_labels == site)[0][:n_traces]
            for index in site_indices:
                traces.append(data[index].tolist())
                labels.append(site_to_id[site])

        return traces, labels

    traces = []
    labels = []
    sites = sorted(entry for entry in resolved_path.iterdir() if entry.is_dir())[:n_sites]

    for label, site in enumerate(sites):
        files = sorted(entry for entry in site.iterdir() if entry.is_file())[:n_traces]
        for trace_file in files:
            directions = []
            with trace_file.open('r', encoding='utf-8', errors='ignore') as handle:
                for line in handle:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        directions.append(int(float(parts[1])))

            if directions:
                traces.append(directions)
                labels.append(label)

    return traces, labels

In [6]:
def burst_direction(value: int) -> str:
    return 'OUT' if value == 1 else 'IN'


def iter_bursts(trace: Sequence[int]) -> Iterator[tuple[str, int]]:
    index = 0

    while index < len(trace):
        current_value = trace[index]
        length = 1

        while index + length < len(trace) and trace[index + length] == current_value:
            length += 1

        yield burst_direction(current_value), length
        index += length


def burst_length_bucket(length: int) -> str:
    if length <= 5:
        return 'XS'
    if length <= 20:
        return 'S'
    if length <= 100:
        return 'M'
    return 'L'


def burst_tokenize(trace: Sequence[int]) -> list[str]:
    return [
        f'{direction}_{burst_length_bucket(length)}'
        for direction, length in iter_bursts(trace)
    ]


def proposal_length_bucket(length: int) -> str:
    if length <= 16:
        return f'LEN_{length:03d}'
    if length <= 32:
        start = ((length - 17) // 2) * 2 + 17
        end = start + 1
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 64:
        start = ((length - 33) // 4) * 4 + 33
        end = start + 3
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 128:
        start = ((length - 65) // 8) * 8 + 65
        end = start + 7
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 256:
        start = ((length - 129) // 16) * 16 + 129
        end = start + 15
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 512:
        start = ((length - 257) // 32) * 32 + 257
        end = start + 31
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 1024:
        start = ((length - 513) // 64) * 64 + 513
        end = start + 63
        return f'LEN_{start:03d}_{end:03d}'
    return 'LEN_1025_PLUS'


def proposal_burst_tokenize(trace: Sequence[int]) -> list[str]:
    return [
        f'{direction}_{proposal_length_bucket(length)}'
        for direction, length in iter_bursts(trace)
    ]


def build_burst_vocabulary(
    traces: Iterable[Sequence[int]],
    tokenize_fn: Callable[[Sequence[int]], list[str]],
) -> set[str]:
    return {
        token
        for trace in traces
        for token in tokenize_fn(trace)
    }

In [7]:
traces, labels = load_traces(DATASET_PATH, n_sites=N_SITES, n_traces=N_TRACES)
print(f'Loaded {len(traces)} traces across {len(set(labels))} sites')
print(f'Sample trace (first 20 cells): {traces[0][:20] if traces else []}')

ValueError: Dataset path does not exist or is unsupported: /content/drive/MyDrive/final project/dataset/tor_100w_2500tr.npz

In [ ]:
coarse_vocab = build_burst_vocabulary(traces, tokenize_fn=burst_tokenize)
proposal_vocab = build_burst_vocabulary(traces, tokenize_fn=proposal_burst_tokenize)

print(f'Coarse sample tokens: {burst_tokenize(traces[0])[:15]}')
print(f'Coarse vocab size: {len(coarse_vocab)}')
print(f'Proposal sample tokens: {proposal_burst_tokenize(traces[0])[:15]}')
print(f'Proposal vocab size: {len(proposal_vocab)}')

Coarse sample tokens: ['OUT_XS', 'IN_M', 'OUT_XS', 'IN_XS', 'OUT_XS', 'IN_M', 'OUT_XS', 'IN_XS', 'OUT_M', 'IN_M', 'OUT_XS', 'IN_M', 'OUT_XS', 'IN_S', 'OUT_XS']
Coarse vocab size: 8
Proposal sample tokens: ['OUT_LEN_002', 'IN_LEN_025_026', 'OUT_LEN_004', 'IN_LEN_001', 'OUT_LEN_002', 'IN_LEN_053_056', 'OUT_LEN_001', 'IN_LEN_002', 'OUT_LEN_021_022', 'IN_LEN_025_026', 'OUT_LEN_003', 'IN_LEN_031_032', 'OUT_LEN_001', 'IN_LEN_014', 'OUT_LEN_002']
Proposal vocab size: 102


In [10]:
from pathlib import Path

print("Colab connected:", IN_COLAB)
print("/content/drive exists:", Path("/content/drive").exists())
print("MyDrive exists:", Path("/content/drive/MyDrive").exists())
print("Project dir exists:", Path("/content/drive/MyDrive/final project/dataset").exists())
print("Dataset exists:", Path("/content/drive/MyDrive/final project/dataset/tor_100w_2500tr.npz").exists())

!find /content/drive/MyDrive -name "tor_100w_2500tr.npz"



Colab connected: True
/content/drive exists: True
MyDrive exists: True
Project dir exists: False
Dataset exists: False
